In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

ad.settings.zarr_write_format = 2

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "01_export_and_annotate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_mouse_preprocessed.zarr")
adata

In [ ]:
def cluster_dotplot(adata, markers, cluster, label=None, block=None, height=100 * fs.MM, width=fs.COL2, brackets=True,
                    dot_max=1, legend_width=1.5):
    obs = adata.obs
    names = list(markers)
    genes = {ct: [g for g in gs if g in adata.var_names] for ct, gs in markers.items()}
    genes = {ct: gs for ct, gs in genes.items() if gs}
    flat = [g for gs in genes.values() for g in gs]

    table = pd.DataFrame({"cluster": obs[cluster].astype(str)})
    table["block"] = obs[block].astype(str) if block is not None else ""
    table = table.drop_duplicates("cluster").set_index("cluster", drop=False).rename_axis(None)
    table["rank"] = table["block"].map(lambda b: names.index(b) if b in names else len(names))
    table["prefix"] = table["cluster"].str.replace(r"\d+$", "", regex=True)
    table["num"] = pd.to_numeric(table["cluster"].str.extract(r"(\d+)$")[0], errors="coerce")
    if label is None:
        x = adata[:, flat].layers["log1p_norm"]
        x = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
        mean = pd.DataFrame(x, columns=flat).groupby(obs[cluster].astype(str).to_numpy()).mean()
        scaled = (mean - mean.min()) / (mean.max() - mean.min())
        score = pd.DataFrame({ct: scaled[gs].mean(axis=1) for ct, gs in genes.items()})
        table["rank"] = score.loc[table.index].to_numpy().argmax(axis=1)
        blocks = list(genes.values())
        table["top_gene"] = [flat.index(scaled.loc[c, blocks[r]].idxmax()) for c, r in zip(table.index, table["rank"])]
        table = table.sort_values(["rank", "top_gene", "prefix", "num", "cluster"])
    else:
        table = table.sort_values(["rank", "prefix", "num", "cluster"])
    rows = list(table["cluster"])

    key = "_cluster_label"
    obs[key] = pd.Categorical(obs[cluster].astype(str).map(dict(zip(table["cluster"], rows))), categories=rows)
    try:
        dp = sc.pl.dotplot(adata, genes if brackets else flat, groupby=key, layer="log1p_norm", standard_scale="var",
                           dot_min=0, dot_max=dot_max,
                           figsize=(width, height), show=False, return_fig=True)
        dp.style(largest_dot=40, dot_edge_lw=0.2)
        dp.legend(width=legend_width, size_title="% cells", colorbar_title="Scaled\nexpression")
        dp.make_figure()
    finally:
        del obs[key]

    ax = dp.get_axes()["mainplot_ax"]
    for t in ax.get_xticklabels():
        t.set_fontstyle("italic")
    ax.set_ylabel("Cluster" if label is None else "Cell type")
    return dp

LINEAGE = {"Epithelium": ["Vil1", "Krt19", "Lyz1", "Muc2", "Tac1"], "Stroma": ["Pdgfra", "Des", "Pecam1"],
           "Immune": ["Ptprc", "Csf1r", "Cd3e", "Jchain"]}
dotplot = cluster_dotplot(adata, LINEAGE, "leiden_rough", height=50 * fs.MM, width=fs.COL1 * 1.3)
fs.save(dotplot.fig, OUT_DIR / "dotplot_lineage")
plt.show()

In [ ]:
EPITHELIUM = ["0", "2", "5", "6", "8"]
NON_EPITHELIUM = ["1", "3", "4", "7", "9", "10"]
assert sorted(EPITHELIUM + NON_EPITHELIUM) == sorted(adata.obs["leiden_rough"].cat.categories)

In [ ]:
sc.tl.leiden(adata, resolution=1.0, restrict_to=('leiden_rough', EPITHELIUM),
            key_added='leiden_mixed')
sc.tl.leiden(adata, 1.0, restrict_to=('leiden_mixed', NON_EPITHELIUM), key_added='leiden_sub')
PREFIX = {"-".join(EPITHELIUM): "E", "-".join(NON_EPITHELIUM): "N"}
short_names = {}
for name in adata.obs["leiden_sub"].cat.categories:
    lineage, number = name.split(",")
    short_names[name] = PREFIX[lineage] + number
adata.obs["leiden"] = adata.obs["leiden_sub"].cat.rename_categories(short_names)
adata.obs["leiden"].value_counts().sort_index()

In [ ]:
mix = pd.crosstab(adata.obs["leiden"], adata.obs["section"], normalize="index").round(3)
mix["cells"] = adata.obs["leiden"].value_counts()
mix.to_csv(OUT_DIR / "leiden_roi_fraction.csv")
mix.loc[mix.drop(columns="cells").max(axis=1).sort_values(ascending=False).index]

In [ ]:
print("Total cells:", adata.n_obs)
adata.obs.groupby(["condition", "slide"], observed=True)["total_counts"].agg(
    n_cells="size", n_transcripts="sum", median_per_cell="median")

In [ ]:
fig, ax = plt.subplots(figsize=(fs.COL2, 40 * fs.MM), layout="constrained")
sc.pl.violin(adata, "total_counts", groupby="leiden", log=True, stripplot=False, ax=ax, show=False)
for violin in ax.collections:
    violin.set_facecolor(fs.LIGHTGREY)
    violin.set_edgecolor(fs.GREY)
    violin.set_linewidth(0.3)
ax.set_xlabel("Cluster")
ax.set_ylabel("Transcripts per cell")
ax.tick_params(axis="x", labelrotation=0)
plt.show()

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", use_raw=True, n_genes=100)
deg = sc.get.rank_genes_groups_df(adata, group=None)
deg.to_csv(OUT_DIR / "rank_genes_leiden.csv", index=False)
deg.groupby("group", observed=True).head(3)

In [ ]:
top10 = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top10.index = range(1, 11)
top10.to_csv(OUT_DIR / "rank_genes_leiden_top10.csv")
top10

In [ ]:
markers = {
    "Enterocyte":           ["Apoa1", "Apob"],
    "Progenitor cell":      ["Mki67", "Top2a"],
    "Paneth cell":          ["Lyz1", "Defa5"],
    "Goblet cell":          ["Muc2", "Spdef"],
    "Enteroendocrine cell": ["Tph1", "Neurod1"],
    "Vascular EC":          ["Plvap", "Kdr"],
    "Lymphatic EC":         ["Lyve1", "Ccl21a"],
    "Muscularis SMC":       ["Des", "Actg2"],
    "Villus SMC":           ["Hhip", "Myh11"],
    "Telocyte":             ["Pdgfra", "Bmp4"],
    "Trophocyte":           ["Grem1", "Rspo3"],
    "Adamdec1+ fibroblast": ["Adamdec1"],
    "Villus fibroblast":    ["Fgfr2", "Col14a1"],
    "Macrophage":           ["Csf1r", "Cd68"],
    "Eosinophil":           ["Alox15", "Ccr3"],
    "Neutrophil":           ["Csf3r", "Acod1"],
    "ILC":                  ["Id2", "Il7r"],
    "T cell":               ["Cd3g", "Cd8a"],
    "Plasma cell":          ["Jchain", "Igkc"],
    "Enteric neuron":       ["Uchl1", "Snap25"],
    "Mesothelial cell":     ["Msln", "Upk3b"],
}

In [ ]:
dotplot = cluster_dotplot(adata, markers, "leiden", height=155 * fs.MM)
fs.save(dotplot.fig, OUT_DIR / "dotplot_markers")
plt.show()

markers_large = {
    "Epithelial cell":  ["Vil1", "Krt19"],
    "Fibroblast":       ["Pdgfra"],
    "Endothelial cell": ["Pecam1"],
    "Muscularis SMC":   ["Des", "Actg2"],
    "Villus SMC":       ["Hhip", "Myh11"],
    "Immune cell":      ["Ptprc", "Csf1r", "Cd3e", "Jchain"],
    "Enteric neuron":   ["Uchl1", "Snap25"],
    "Mesothelial cell": ["Msln", "Upk3b"],
}
dotplot = cluster_dotplot(adata, markers_large, "leiden", height=155 * fs.MM, width=fs.COL1 * 1.5)
fs.save(dotplot.fig, OUT_DIR / "dotplot_markers_large")
plt.show()

In [ ]:
subtype_clusters = {
    "Enterocyte":           ["E1", "E2", "E3", "E4", "E6", "E7", "E8", "E10", "E11", "E12", "E13", "E14", "E15", "E16", "E17", "E18", "E19", "E20", "E21", "E22", "E23", "E24", "E25", "E27", "E28", "E29", "E30", "E31", "E32"],
    "Progenitor cell":      ["E0"],
    "Paneth cell":          ["E5"],
    "Goblet cell":          ["E9"],
    "Enteroendocrine cell": ["E26"],
    "Vascular EC":          ["N3", "N5"],
    "Lymphatic EC":         ["N14"],
    "Muscularis SMC":       ["N0"],
    "Villus SMC":           ["N13"],
    "Telocyte":             ["N10"],
    "Trophocyte":           ["N11"],
    "Adamdec1+ fibroblast": ["N4", "N7"],
    "Villus fibroblast":    ["N6"],
    "Macrophage":           ["N2"],
    "Eosinophil":           ["N12"],
    "Neutrophil":           ["N17"],
    "ILC":                  ["N8"],
    "T cell":               ["N9"],
    "Plasma cell":          ["N1"],
    "Enteric neuron":       ["N15"],
    "Mesothelial cell":     ["N16"],
    "Unassigned":           ["N18", "N19", "N20"],
}
adata.obs["cell_subtype"] = ""
for subtype, clusters in subtype_clusters.items():
    adata.obs.loc[adata.obs["leiden"].isin(clusters), "cell_subtype"] = subtype
adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"], categories=list(subtype_clusters))

largetype_subtypes = {
    "Epithelial cell":  ["Enterocyte", "Progenitor cell", "Paneth cell", "Goblet cell", "Enteroendocrine cell"],
    "Fibroblast":       ["Telocyte", "Trophocyte", "Adamdec1+ fibroblast", "Villus fibroblast"],
    "Endothelial cell": ["Vascular EC", "Lymphatic EC"],
    "Muscularis SMC":   ["Muscularis SMC"],
    "Villus SMC":       ["Villus SMC"],
    "Immune cell":      ["Macrophage", "Eosinophil", "Neutrophil", "ILC", "T cell", "Plasma cell"],
    "Enteric neuron":   ["Enteric neuron"],
    "Mesothelial cell": ["Mesothelial cell"],
    "Unassigned":       ["Unassigned"],
}
adata.obs["cell_largetype"] = ""
for largetype, subtypes in largetype_subtypes.items():
    adata.obs.loc[adata.obs["cell_subtype"].isin(subtypes), "cell_largetype"] = largetype
adata.obs["cell_largetype"] = pd.Categorical(adata.obs["cell_largetype"], categories=list(largetype_subtypes))

pd.crosstab(adata.obs["cell_subtype"], [adata.obs["condition"], adata.obs["slide"]])

In [ ]:
FIBROBLASTS = ["Telocyte", "Trophocyte", "Adamdec1+ fibroblast", "Villus fibroblast"]
fibroblasts = adata[adata.obs["cell_subtype"].isin(FIBROBLASTS)].copy()
fibroblasts.obs["cell_subtype"] = fibroblasts.obs["cell_subtype"].cat.remove_unused_categories()
dotplot = cluster_dotplot(fibroblasts, {celltype: markers[celltype] for celltype in FIBROBLASTS}, "cell_subtype", label="cell_subtype", block="cell_subtype",
                        height=40 * fs.MM, width=110 * fs.MM, brackets=False)
fs.save(dotplot.fig, OUT_DIR / "dotplot_fibroblast")
plt.show()

In [ ]:
def umap(ax, xy, labels, size=0.3):
    labels = pd.Categorical(labels).remove_unused_categories()
    cats = labels.categories
    order = np.random.default_rng(0).permutation(len(xy))
    colors = np.array(fs.palette(labels.astype(str)))
    ax.scatter(xy[order, 0], xy[order, 1], c=colors[order], s=size, lw=0, rasterized=True)
    ax.set_aspect("equal")
    ax.set_axis_off()

    handles = [mpl.lines.Line2D([], [], ls="", marker="o", ms=3, mew=0, color=c) for c in fs.palette(cats)]
    ax.legend(handles, cats, loc="center left", bbox_to_anchor=(1.0, 0.5), borderaxespad=0, handletextpad=0.1,
              labelspacing=0.25)

    span = np.ptp(xy, axis=0).max()
    x0, y0 = xy.min(axis=0) - 0.03 * span
    arrow = dict(arrowstyle="-|>", lw=0.5, color=fs.BLACK, mutation_scale=4, shrinkA=0, shrinkB=0)
    ax.update_datalim([(x0, y0)])
    ax.annotate("", (x0 + 0.15 * span, y0), (x0, y0), arrowprops=arrow)
    ax.annotate("", (x0, y0 + 0.15 * span), (x0, y0), arrowprops=arrow)
    ax.annotate("UMAP1", (x0 + 0.075 * span, y0), xytext=(0, -2), textcoords="offset points", fontsize=5,
                ha="center", va="top")
    ax.annotate("UMAP2", (x0, y0 + 0.075 * span), xytext=(-2, 0), textcoords="offset points", fontsize=5,
                ha="right", va="center", rotation=90)
    return ax

def save_umap(color, name):
    fig, ax = plt.subplots(figsize=(1.5 * fs.COL1, 0.85 * fs.COL1), layout="constrained")
    umap(ax, adata.obsm["X_umap"], adata.obs[color], size=0.2)
    fs.save(fig, OUT_DIR / name)
    plt.show()

save_umap("cell_subtype", "umap_cell_subtype")
save_umap("cell_largetype", "umap_cell_largetype")

In [ ]:
def composition(obs, group, stack, groups, title=None, width=fs.COL2, height=55 * fs.MM, panels=("count", "fraction"),
                rows=None, row_labels=None, tick_label=None):
    ct = pd.crosstab(obs[group], obs[stack]).reindex(index=groups, fill_value=0)
    ct = ct.loc[:, ct.sum() > 0]
    frac = ct.div(ct.sum(axis=1), axis=0) * 100
    spec = {"count": (ct, "Number of cells"), "fraction": (frac, "Proportion of cells (%)")}
    rows = rows or [groups]
    fig, axes = plt.subplots(len(rows), len(panels), figsize=(width, height), layout="constrained", squeeze=False,
                             sharey="col")
    for r, row in enumerate(rows):
        for ax, panel in zip(axes[r], panels):
            table, ylabel = spec[panel]
            table = table.loc[row]
            xs = np.arange(len(table))
            bottom = np.zeros(len(table))
            for c, color in zip(table.columns, fs.palette(table.columns)):
                ax.bar(xs, table[c], bottom=bottom, width=0.75, color=color, edgecolor="white", linewidth=0.25, label=c)
                bottom += table[c].to_numpy(dtype=float)
            if tick_label is None:
                ax.set_xticks(xs, [fs.label(g) for g in table.index], rotation=45, ha="right", rotation_mode="anchor")
            else:
                ax.set_xticks(xs, [tick_label(g) for g in table.index])
            if row_labels is not None:
                ax.set_title(row_labels[r])
            ax.set_ylabel(ylabel)
            if panel == "count":
                ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
            else:
                ax.set_ylim(0, 100)
    handles, names = axes[0, -1].get_legend_handles_labels()
    fig.legend(handles[::-1], names[::-1], title=title, loc="outside right center", alignment="left")
    return fig, ct

SECTIONS = list(adata.obs["section"].cat.categories)
GROUPS = ["C-ion", "γ-ray"]
ROWS = [[section for section in SECTIONS if section.startswith(radiation)] for radiation in GROUPS]
TICK_LABELS = {section: section.split(" ", 1)[1].replace(" (", "\n(") for section in SECTIONS}

def save_composition(cells, stack, name):
    for panel in ["count", "fraction"]:
        fig, counts = composition(cells, "section", stack, SECTIONS, width=0.6 * fs.COL2, height=75 * fs.MM,
                                 panels=(panel,), rows=ROWS, row_labels=GROUPS, tick_label=TICK_LABELS.get)
        fs.save(fig, OUT_DIR / f"composition_{name}_{panel}")
        plt.show()
    counts.to_csv(OUT_DIR / f"composition_{name}.csv")

obs = adata.obs
save_composition(obs, "cell_largetype", "cell_largetype")
save_composition(obs[obs["cell_largetype"] != "Epithelial cell"], "cell_subtype", "non_epithelial")
save_composition(obs[obs["cell_largetype"] == "Fibroblast"], "cell_subtype", "fibroblast")
save_composition(obs[obs["cell_largetype"] == "Immune cell"], "cell_subtype", "immune")

In [ ]:
for key in [key for key in adata.uns if key.endswith("_colors")]:
    del adata.uns[key]
adata.write_zarr(BASE / "data" / "integrated_mouse_annotated.zarr")